<a href="https://colab.research.google.com/github/Mauricio-Lopez1458/assistente-bradesco-dio/blob/main/virtual_assistant_bradesco_dio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = "pt"

1. Apresentação em áudio da assistente virtual (gTTS)

In [ ]:
#Instalando o Google Text To Speech (gTTS)
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
#gravando o áudio da apresentação.
from gtts import gTTS
from IPython.display import Audio, display

# texto da apresentação a ser convertido em áudio)
text_to_speak = """Oi, eu sou a Bri, prima da Bia. Bri significa Bradesco Investimentos.
Estou aqui para te ajudar a fazer a melhor escolha em investimentos no Bradesco.
Sobre qual tipo de investimento você deseja informações?
Fale em poucas palavras.
                """

# criando um objeto gTTS
tts = gTTS(text=text_to_speak, lang=language, slow=False) # 'slow=False' sets a normal speed

# salva o texto (convertido em áudio) em arquivo .wav
sound_file1 = "/content/assistant_presentation.wav"
tts.save(sound_file1)

# mostra o player de áudio
display(Audio(sound_file1, autoplay=True)) # 'autoplay=True' will play the sound automatically


2. Gravação de Áudio com Python (e Um Pouco de Javascript)

In [ ]:
# @title
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

#o código abaixo (contido na variável 'RECORD') é um Web API para captar áudio do microfone do usuário de qualquer navegador que ele estiver usando.
#o código abaixo está em Javascript (dentro da variável 'RECORD' da linguagem Python).
RECORD = """
const sleep = time => new Promise(resolve=> setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=> {
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""


#função abaixo resgata a gravação feita no código acima em Javascript (apenas 6 segundos de gravação)
def record(sec=6):
  display(Javascript(RECORD)) #fazendo o código Javascript (dentro da variável 'RECORD') ser interpretado no código em Python.
  js_result = output.eval_js('record(%s)' % (sec * 1000)) #executa o código em Javascript acima e retorna a resposta (executa o Javascript dentro de um código Python).
  audio = b64decode(js_result.split(',')[1])#decodifica o arquivo de áudio gerado na captação com Javascript e que, por padrão, é o segundo elemento do resultado (índice 1).
  file_name = 'request_audio.wav' #nome criado do arquivo onde será escrito (gravado) o áudio captado anteriormente.
  #o código abaixo abre um arquivo (usando a função 'open()') onde será escrito (gravado) o áudio captado anteriormente (o 'wb' indica que o áudio será em arquivo binário).
  with open(file_name, 'wb') as f:
    f.write(audio)#a função 'write()' grava (escreve) o áudio captado no arquivo 'request_audio.wav'.
  return f'/content/{file_name}' #indica que o arquivo de áudio salvo ficará na pasta raiz do Colab ('sample_data').

print('Diga alguma coisa...\n')
record_file = record() #chamando a função 'record' para executar todo o código (incluindo o código em Javascript) e assim captando áudio do usuário e salvando em um arquivo.
display(Audio(record_file, autoplay=True)) #exibirá na tela um player de áudio que executará o áudio captado e salvo, assim que acabar a captação.



Diga alguma coisa...



<IPython.core.display.Javascript object>

3. Reconhecimento de Fala com Whisper (OpenAI)

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q #instalando a biblioteca do Whisper.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper #import a biblioteca 'whisper' (speech-to-text) instalada na célula anterior.
model = whisper.load_model("small") #a variável 'modelo' carrega o whisper e especifica o tipo "small" para o projeto (poderia ser "medium" ou "large")
result = model.transcribe(record_file, fp16=False, language=language) #utilizando o whisper, 'result' faz a transcrição do arquivo de áudio 'record_file'.
transcription = result["text"] #resgata o texto da transcrição do áudio.
print(transcription)

 Letras de câmbio.


4. Integração com a API do Gemini

In [ ]:
#Instalando Python SDK para poder acessar o Gemini API (está no pacote 'google-generativeai').
!pip install -q -U google-generativeai

In [ ]:
#Importando os pacotes do Gemini API
import pathlib
import textwrap

import google.generativeai as genai

from IPython.display import display
from IPython.display import Markdown


def to_markdown(text):
    text = text.replace("•", "  *")
    return Markdown(textwrap.indent(text, "> ", predicate=lambda _: True))

In [ ]:
#Configurando e aplicando uma API key (uma API key pode ser conseguida no Google AI Studio)
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
#Gerando texto do input (texto de resposta do Gemini).
#Usando o modelo "Gemini 2.5 Flash"
model = genai.GenerativeModel("gemini-2.5-flash")
#Gerando conteúdo
gemini_response = model.generate_content(f"""Dentro do escopo de Bradesco Investimentos, fale em poucas palavras sobre {transcription}.
Caso {transcription} não esteja relacionado a Bradesco Investimentos, retorne a frase 'Desculpe, esse assunto não está relacionado a investimentos.'""")
generated_text = gemini_response.text.strip()#colocando o conteúdo gerado em um objeto ('generated_text') para poder transformar em voz depois.
to_markdown(generated_text)

> Dentro do escopo de Bradesco Investimentos, Letras de Câmbio (LCs) são títulos de renda fixa emitidos por financeiras (não por bancos comerciais). Representam um investimento que remunera o aplicador com juros, sendo uma alternativa de dívida privada para diversificação e normalmente contam com a proteção do FGC (Fundo Garantidor de Créditos).

5. Sintetizando a Resposta do Gemini como Voz (gTTS)

In [ ]:
#Instalando o Google Text To Speech (gTTS)
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
#Integrando o texto gerado pelo Gemini com o Google Text To Speech (transformando em voz)
from gtts import gTTS

#Cria um objeto (resultado em voz) e especifica:
# - o arquivo de texto (gerado pelo Gemini) a ser convertido em voz (text).
# - o idioma a ser usado (lang)
# - a velocidade da fala - se é lent aou não (slow)
gtts_object = gTTS(text=generated_text, lang=language, slow=False)
response_audio = "/content/response_audio.wav"#Cria o arquivo de áudio onde será armazenada a voz gerada sobre o texto fornecido.
gtts_object.save(response_audio) #salva o áudio gerado no arquivo response audio.
display(Audio(response_audio, autoplay=True))# Cria um player de áudio que executará o arquivo ('autoplay') assim que for gerado.